# Piloto: monitoramento de desmatamento com Sentinel-1 (SAR) e Sentinel-2 (óptico)

Desenvolvi este estudo de caso pessoal ao longo de um fim de semana para demonstrar, de forma objetiva, meu interesse em contribuir com um time de visão computacional aplicada ao sensoriamento remoto.

**Pergunta do piloto:** quanto o radar (SAR) acrescenta à imagem óptica na detecção de desmatamento na Amazônia, onde a nuvem cobre boa parte do ano?

**O que este notebook faz**

1. Escolhe uma área-piloto de cerca de 27 × 27 km no arco do desmatamento.
2. Monta composições de estação seca **antes** e **depois** com Sentinel-2 (máscara de nuvem Cloud Score+) e Sentinel-1 (VV e VH).
3. Mede, mês a mês, quanto da área o óptico e o SAR conseguem observar.
4. Usa como rótulo a perda florestal do Hansen Global Forest Change (v1.12), só dentro da floresta remanescente no início do período.
5. Treina três Random Forests (só óptico, só SAR, fusão) numa metade da área e avalia **todos os pixels** da outra metade (holdout espacial).
6. Converte a detecção em **alertas georreferenciados** (polígonos com área em hectares) e gera um mapa HTML compartilhável.
7. Gera imagens de antes e depois e um `README.md` já preenchido com os resultados.

**Como rodar:** configure a variável `EE_PROJECT` com o ID do seu projeto Google Cloud e use *Executar tudo*. Tempo típico: 10 a 25 minutos.

Autor: Laércio Siqueira

## 0. Instalação e imports

In [ ]:
# Funciona no Colab e no Jupyter local (VS Code). O Colab já traz quase tudo.
%pip install -q -U earthengine-api
%pip install -q folium geopandas matplotlib pillow requests

In [ ]:
import io
import json
import math
import zipfile

import ee
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
from shapely.geometry import mapping, shape


def feicao(geom):
    # Converte uma ee.Geometry num Feature GeoJSON para o folium
    return {"type": "Feature", "geometry": geom.getInfo(), "properties": {}}

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

## 1. Autenticação no Earth Engine

Configure a variável de ambiente `EE_PROJECT` com o ID de um projeto Google Cloud autorizado para uso no Earth Engine. A célula usa autenticação local com escopo somente de leitura e não grava o ID do projeto no notebook.

Na primeira execução, o navegador abrirá para você confirmar a conta Google e conceder a autorização.

In [ ]:
import os
import ee

PROJECT_ID = os.environ.get("EE_PROJECT", "seu-projeto-aqui")

assert PROJECT_ID != "seu-projeto-aqui", "Defina EE_PROJECT ou troque PROJECT_ID pelo ID do seu projeto."
ee.Authenticate(
    auth_mode="localhost",
    scopes=["https://www.googleapis.com/auth/earthengine.readonly"],
)
ee.Initialize(project=PROJECT_ID)
print("Earth Engine conectado.")

## 2. Parâmetros do piloto

Tudo que você talvez queira ajustar fica aqui. Os valores padrão já funcionam.

In [ ]:
ANO_ROTULO = 2023          # ano da perda florestal usada como rótulo (banda lossyear do Hansen)
ESCALA = 30                # resolução de trabalho em metros (a mesma do rótulo)
CS_LIMIAR = 0.60           # Cloud Score+: pixel "limpo" se cs_cdf >= limiar (0,50 a 0,65 costuma funcionar)
COPA_MIN = 30              # % mínima de copa em 2000 para considerar floresta
LIMIAR_PROB = 0.5          # probabilidade mínima para marcar desmatamento
AREA_MIN_HA = 1.0          # área mínima de um alerta, em hectares
N_AMOSTRAS_CLASSE = 2000   # pontos de treino por classe
N_ARVORES = 150            # árvores do Random Forest
SEED = 42
LADO_GRAUS = 0.25          # lado da área-piloto, em graus (cerca de 27 km)

# Janelas de estação seca: antes (ano anterior ao rótulo) e depois (ano do rótulo)
T0 = (f"{ANO_ROTULO - 1}-06-01", f"{ANO_ROTULO - 1}-10-31")
T1 = (f"{ANO_ROTULO}-06-01", f"{ANO_ROTULO}-10-31")

# Áreas candidatas no arco do desmatamento (lon, lat do centro).
# O notebook escolhe a que tem mais perda florestal rotulada no ano.
CANDIDATAS = {
    "Novo Progresso (PA)": (-55.40, -7.15),
    "Apuí (AM)": (-59.75, -7.45),
    "São Félix do Xingu (PA)": (-52.30, -6.40),
    "Castelo dos Sonhos, Altamira (PA)": (-55.00, -8.20),
    "Colniza (MT)": (-59.00, -9.40),
}

# Para fixar uma área específica, preencha com o nome de uma candidata (ou deixe None).
AOI_FIXA = None

print("Janela antes (T0):", T0)
print("Janela depois (T1):", T1)

## 3. Funções auxiliares e rótulos

In [ ]:
def caixa(lon, lat, lado=LADO_GRAUS):
    h = lado / 2
    return ee.Geometry.Rectangle([lon - h, lat - h, lon + h, lat + h])


def epsg_utm(lon, lat):
    zona = int((lon + 180) // 6) + 1
    return f"EPSG:{(32700 if lat < 0 else 32600) + zona}"


GFC = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")


def rotulos(ano=ANO_ROTULO):
    # 1 = perda florestal no ano; 0 = floresta que permaneceu.
    # Mascarado (fora do domínio) onde não havia floresta no início do período,
    # como faz o PRODES ao monitorar só a floresta remanescente.
    yy = ano - 2000
    perda = GFC.select("lossyear")
    terra = GFC.select("datamask").eq(1)
    floresta_t0 = GFC.select("treecover2000").gte(COPA_MIN).And(perda.eq(0).Or(perda.gte(yy)))
    return perda.eq(yy).rename("label").updateMask(floresta_t0.And(terra))


AREA_HA = ee.Image.pixelArea().divide(1e4)

## 4. Escolha da área-piloto

Para cada candidata, calcula a área de floresta remanescente e a área de perda florestal no ano do rótulo.

In [ ]:
label_global = rotulos()
img_areas = (AREA_HA.updateMask(label_global.eq(1)).rename("desmat_ha")
             .addBands(AREA_HA.updateMask(label_global.gte(0)).rename("floresta_ha")))

fc_cand = ee.FeatureCollection([
    ee.Feature(caixa(lon, lat), {"nome": nome}) for nome, (lon, lat) in CANDIDATAS.items()
])
res = img_areas.reduceRegions(collection=fc_cand, reducer=ee.Reducer.sum(),
                              scale=ESCALA, tileScale=4).getInfo()
df_cand = (pd.DataFrame([f["properties"] for f in res["features"]])
           .reindex(columns=["nome", "desmat_ha", "floresta_ha"]).fillna(0).sort_values("desmat_ha", ascending=False).reset_index(drop=True))
df_cand["perda_%_da_floresta"] = 100 * df_cand["desmat_ha"] / df_cand["floresta_ha"].replace(0, np.nan)
display(df_cand.round(1))

NOME_AOI = AOI_FIXA or df_cand.loc[0, "nome"]
LON, LAT = CANDIDATAS[NOME_AOI]
AOI = caixa(LON, LAT)
CRS = epsg_utm(LON, LAT)
print(f"Área-piloto: {NOME_AOI} | centro ({LON}, {LAT}) | projeção de trabalho {CRS}")

desmat_aoi = float(df_cand.loc[df_cand["nome"] == NOME_AOI, "desmat_ha"].iloc[0])
if desmat_aoi < 300:
    print("ATENÇÃO: pouca perda rotulada nesta área. Considere trocar ANO_ROTULO ou as coordenadas das candidatas.")

### Divisão espacial treino/teste

Pixels vizinhos são parecidos, então sortear pixels para treino e teste infla as métricas. Aqui o treino usa uma metade da área e o teste usa a outra. O notebook escolhe o corte (leste-oeste ou norte-sul) que deixa as duas metades com mais desmatamento rotulado.

In [ ]:
h = LADO_GRAUS / 2
metades = {
    "oeste": ee.Geometry.Rectangle([LON - h, LAT - h, LON, LAT + h]),
    "leste": ee.Geometry.Rectangle([LON, LAT - h, LON + h, LAT + h]),
    "sul": ee.Geometry.Rectangle([LON - h, LAT - h, LON + h, LAT]),
    "norte": ee.Geometry.Rectangle([LON - h, LAT, LON + h, LAT + h]),
}
fc_metades = ee.FeatureCollection([ee.Feature(g, {"metade": k}) for k, g in metades.items()])
res_m = img_areas.select("desmat_ha").reduceRegions(collection=fc_metades, reducer=ee.Reducer.sum(),
                                                     scale=ESCALA, tileScale=4).getInfo()
desm = {f["properties"]["metade"]: f["properties"].get("sum", 0) or 0 for f in res_m["features"]}

if min(desm["oeste"], desm["leste"]) >= min(desm["sul"], desm["norte"]):
    TREINO, TESTE, NOME_TREINO, NOME_TESTE = metades["oeste"], metades["leste"], "oeste", "leste"
else:
    TREINO, TESTE, NOME_TREINO, NOME_TESTE = metades["sul"], metades["norte"], "sul", "norte"

print({k: round(v) for k, v in desm.items()}, "(ha de perda rotulada por metade)")
print(f"Treino na metade {NOME_TREINO}, teste na metade {NOME_TESTE}.")

## 5. Composições de Sentinel-2 e Sentinel-1

- **Sentinel-2:** reflectância de superfície, nuvem e sombra removidas com Cloud Score+, mediana da janela. Índices: NDVI (vegetação), NBR (queimada e solo exposto) e NDMI (umidade da vegetação).
- **Sentinel-1:** retroespalhamento VV e VH em dB, mediana da janela, uma única direção de órbita para evitar misturar ângulos de visada. A razão VH − VV (em dB) ajuda a separar floresta de área aberta.

In [ ]:
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
CSPLUS = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
BANDAS_S2 = ["B2", "B3", "B4", "B8", "B11", "B12"]

S1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))


def colecao_s2(inicio, fim):
    return S2.filterBounds(AOI).filterDate(inicio, fim).linkCollection(CSPLUS, ["cs_cdf"])


def mascara_nuvem(img):
    return img.updateMask(img.select("cs_cdf").gte(CS_LIMIAR))


def composicao_s2(inicio, fim, suf):
    col = colecao_s2(inicio, fim).map(mascara_nuvem).select(BANDAS_S2)
    img = col.median().divide(10000)
    ndvi = img.normalizedDifference(["B8", "B4"]).rename(f"ndvi_{suf}")
    nbr = img.normalizedDifference(["B8", "B12"]).rename(f"nbr_{suf}")
    ndmi = img.normalizedDifference(["B8", "B11"]).rename(f"ndmi_{suf}")
    rgb = img.select(["B4", "B3", "B2"]).rename([f"r_{suf}", f"g_{suf}", f"b_{suf}"])
    return ee.Image.cat([ndvi, nbr, ndmi, rgb]), col.size()


def colecao_s1(inicio, fim, passagem=None):
    col = S1.filterBounds(AOI).filterDate(inicio, fim)
    if passagem is not None:
        col = col.filter(ee.Filter.eq("orbitProperties_pass", passagem))
    return col.select(["VV", "VH"])


# Escolhe a direção de órbita com mais imagens nas duas janelas
contagens = ee.Dictionary({
    p: ee.List([colecao_s1(*T0, p).size(), colecao_s1(*T1, p).size()])
    for p in ["ASCENDING", "DESCENDING"]
}).getInfo()
PASSAGEM = max(contagens, key=lambda p: min(contagens[p]))
if min(contagens[PASSAGEM]) == 0:
    PASSAGEM = None  # nenhuma direção cobre as duas janelas sozinha: usa as duas
print("Imagens Sentinel-1 por direção (T0, T1):", contagens, "| usando:", PASSAGEM or "ambas")


def composicao_s1(inicio, fim, suf):
    col = colecao_s1(inicio, fim, PASSAGEM)
    img = col.median()
    vv = img.select("VV").rename(f"vv_{suf}")
    vh = img.select("VH").rename(f"vh_{suf}")
    razao = vh.subtract(vv).rename(f"razao_{suf}")
    return ee.Image.cat([vv, vh, razao]), col.size()


s2_t0, n_s2_t0 = composicao_s2(*T0, "t0")
s2_t1, n_s2_t1 = composicao_s2(*T1, "t1")
s1_t0, n_s1_t0 = composicao_s1(*T0, "t0")
s1_t1, n_s1_t1 = composicao_s1(*T1, "t1")

n_imagens = ee.Dictionary({"S2 T0": n_s2_t0, "S2 T1": n_s2_t1, "S1 T0": n_s1_t0, "S1 T1": n_s1_t1}).getInfo()
print("Imagens usadas em cada composição:", n_imagens)
assert min(n_imagens.values()) > 0, "Alguma janela ficou sem imagens: revise datas ou área."

## 6. Variáveis (features)

In [ ]:
def deltas(img_t0, img_t1, nomes):
    return ee.Image.cat([
        img_t1.select(f"{n}_t1").subtract(img_t0.select(f"{n}_t0")).rename(f"d_{n}") for n in nomes
    ])


d_s2 = deltas(s2_t0, s2_t1, ["ndvi", "nbr", "ndmi"])
d_s1 = deltas(s1_t0, s1_t1, ["vv", "vh", "razao"])

FEAT_S2 = [f"{n}_{t}" for n in ["ndvi", "nbr", "ndmi"] for t in ["t0", "t1"]] + ["d_ndvi", "d_nbr", "d_ndmi"]
FEAT_S1 = [f"{n}_{t}" for n in ["vv", "vh", "razao"] for t in ["t0", "t1"]] + ["d_vv", "d_vh", "d_razao"]
FEAT_FUSAO = FEAT_S2 + FEAT_S1

FEATURES = ee.Image.cat([s2_t0, s2_t1, d_s2, s1_t0, s1_t1, d_s1]).clip(AOI)
LABEL = rotulos().clip(AOI)
DOMINIO = LABEL.gte(0)  # floresta remanescente no início do período

print(len(FEAT_S2), "variáveis ópticas |", len(FEAT_S1), "variáveis SAR |", len(FEAT_FUSAO), "na fusão")

## 7. Por que o SAR importa: cobertura mensal de observação

Para cada mês do ano do rótulo, mede a fração da área-piloto com **pelo menos uma observação útil**: no óptico, um pixel sem nuvem; no SAR, qualquer passagem. É o gráfico que abre a conversa.

In [ ]:
S2_BASE = S2.filterBounds(AOI)
S1_BASE = S1.filterBounds(AOI)


def cobertura_mes(m):
    m = ee.Number(m)
    ini = ee.Date.fromYMD(ANO_ROTULO, m, 1)
    fim = ini.advance(1, "month")
    s2m = S2_BASE.filterDate(ini, fim).linkCollection(CSPLUS, ["cs_cdf"])
    s2_limpo = ee.Image(ee.Algorithms.If(
        s2m.size().gt(0),
        s2m.map(lambda img: img.select("cs_cdf").gte(CS_LIMIAR)).max().rename("v"),
        ee.Image(0).rename("v")))
    s1m = S1_BASE.filterDate(ini, fim)
    s1_obs = ee.Image(ee.Algorithms.If(
        s1m.size().gt(0),
        s1m.select("VV").count().gt(0).rename("v"),
        ee.Image(0).rename("v")))
    reduz = dict(reducer=ee.Reducer.mean(), geometry=AOI, scale=300, maxPixels=1e9)
    return ee.Feature(None, {
        "mes": m,
        "optico": s2_limpo.unmask(0).reduceRegion(**reduz).get("v"),
        "sar": s1_obs.unmask(0).reduceRegion(**reduz).get("v"),
    })


cob = ee.FeatureCollection(ee.List.sequence(1, 12).map(cobertura_mes)).getInfo()
df_cob = pd.DataFrame([f["properties"] for f in cob["features"]]).sort_values("mes").reset_index(drop=True)
df_cob[["optico", "sar"]] = df_cob[["optico", "sar"]].astype(float).fillna(0) * 100

MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(MESES, df_cob["sar"], marker="o", lw=2, color="#2a6f97", label="SAR (Sentinel-1)")
ax.plot(MESES, df_cob["optico"], marker="o", lw=2, color="#e07a1f", label="Óptico sem nuvem (Sentinel-2)")
ax.set_ylim(0, 105)
ax.set_ylabel("% da área observada no mês")
ax.set_title(f"O SAR observa a área todo mês; o óptico depende da nuvem ({ANO_ROTULO})", fontsize=11, loc="left")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("cobertura_mensal.png", bbox_inches="tight")
plt.show()

PIOR_MES = MESES[int(df_cob.loc[df_cob["optico"].idxmin(), "mes"]) - 1]
print(f"Pior mês para o óptico: {PIOR_MES} ({df_cob['optico'].min():.0f}% da área com observação limpa)")

## 8. Amostragem de treino (metade de treino)

Amostragem estratificada e balanceada: o mesmo conjunto de pontos treina os três modelos, para a comparação ser justa.

In [ ]:
pilha = FEATURES.select(FEAT_FUSAO).addBands(LABEL)
amostras_treino = pilha.stratifiedSample(
    numPoints=N_AMOSTRAS_CLASSE, classBand="label", region=TREINO,
    scale=ESCALA, projection=ee.Projection(CRS), seed=SEED,
    classValues=[0, 1], classPoints=[N_AMOSTRAS_CLASSE, N_AMOSTRAS_CLASSE],
    tileScale=4, geometries=True)

hist_treino = amostras_treino.aggregate_histogram("label").getInfo()
print("Pontos de treino por classe (0 = floresta, 1 = desmatamento):", hist_treino)

## 9. Três modelos e avaliação exaustiva na metade de teste

A avaliação não usa amostra: compara a predição com o rótulo em **todos os pixels de floresta** da metade de teste. Assim as métricas refletem a proporção real entre desmatamento e floresta.

In [ ]:
def treinar(bandas):
    return (ee.Classifier.smileRandomForest(numberOfTrees=N_ARVORES, minLeafPopulation=2, seed=SEED)
            .setOutputMode("MULTIPROBABILITY")
            .train(features=amostras_treino, classProperty="label", inputProperties=bandas))


def probabilidade(clf, bandas):
    return (FEATURES.select(bandas).classify(clf).arrayGet([1])
            .rename("prob").updateMask(DOMINIO))


def histograma_confusao(prob):
    pred = prob.gte(LIMIAR_PROB).toInt()
    comb = LABEL.multiply(2).add(pred).toInt().rename("c")  # 0=VN, 1=FP, 2=FN, 3=VP
    return comb.reduceRegion(reducer=ee.Reducer.frequencyHistogram(), geometry=TESTE,
                             crs=CRS, scale=ESCALA, maxPixels=1e10, tileScale=4).get("c")


def metricas(hist, n_dominio):
    hist = hist or {}
    cont = {int(float(k)): v for k, v in hist.items()}
    vn, fp, fn, vp = (cont.get(i, 0) for i in range(4))
    precisao = vp / (vp + fp) if (vp + fp) else float("nan")
    recall = vp / (vp + fn) if (vp + fn) else float("nan")
    f1 = (2 * precisao * recall / (precisao + recall)
          if (vp and (precisao + recall)) else float("nan"))
    ha = ESCALA ** 2 / 1e4
    return {
        "precisao": precisao, "recall": recall, "f1": f1,
        "cobertura": (vn + fp + fn + vp) / n_dominio if n_dominio else float("nan"),
        "vp": vp, "fp": fp, "fn": fn, "vn": vn,
        "desmat_referencia_ha": (vp + fn) * ha, "desmat_detectado_ha": (vp + fp) * ha,
    }


n_dominio_teste = LABEL.reduceRegion(reducer=ee.Reducer.count(), geometry=TESTE, crs=CRS,
                                     scale=ESCALA, maxPixels=1e10, tileScale=4).get("label").getInfo()

MODELOS = {"Só óptico (S2)": FEAT_S2, "Só SAR (S1)": FEAT_S1, "Fusão S1+S2": FEAT_FUSAO}
CLASSIFICADORES, PROBS, RESULTADOS = {}, {}, {}
for nome, bandas in MODELOS.items():
    clf = treinar(bandas)
    prob = probabilidade(clf, bandas)
    CLASSIFICADORES[nome], PROBS[nome] = clf, prob
    RESULTADOS[nome] = metricas(histograma_confusao(prob).getInfo(), n_dominio_teste)
    print(f"{nome}: F1 = {RESULTADOS[nome]['f1']:.3f}")

df_res = pd.DataFrame(RESULTADOS).T.astype(float)
display(df_res[["precisao", "recall", "f1", "cobertura", "desmat_referencia_ha", "desmat_detectado_ha"]].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
metricas_plot = ["precisao", "recall", "f1"]
rotulos_plot = ["Precisão", "Recall", "F1"]
cores = {"Só óptico (S2)": "#e07a1f", "Só SAR (S1)": "#2a6f97", "Fusão S1+S2": "#3a7d44"}
x = np.arange(len(metricas_plot))
larg = 0.26
for i, (nome, linha) in enumerate(df_res.iterrows()):
    vals = [linha[m] for m in metricas_plot]
    barras = ax.bar(x + (i - 1) * larg, vals, larg, label=nome, color=cores[nome])
    ax.bar_label(barras, fmt="%.2f", fontsize=8, padding=2)
ax.set_xticks(x, rotulos_plot)
ax.set_ylim(0, 1.08)
ax.set_title(f"Desempenho na metade de teste ({NOME_TESTE}), todos os pixels de floresta", fontsize=11, loc="left")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.savefig("comparacao_modelos.png", bbox_inches="tight")
plt.show()

## 10. O que o modelo de fusão usa (importância das variáveis)

In [ ]:
imp = CLASSIFICADORES["Fusão S1+S2"].explain().get("importance").getInfo()
df_imp = pd.Series(imp).sort_values()
df_imp = 100 * df_imp / df_imp.sum()
cor_barra = ["#2a6f97" if n in FEAT_S1 else "#e07a1f" for n in df_imp.index]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(df_imp.index, df_imp.values, color=cor_barra)
ax.set_xlabel("% da importância total")
ax.set_title("Importância das variáveis na fusão (azul = SAR, laranja = óptico)", fontsize=11, loc="left")
plt.tight_layout()
plt.savefig("importancia_variaveis.png", bbox_inches="tight")
plt.show()

print(f"Participação do SAR na importância total: {df_imp[[n for n in df_imp.index if n in FEAT_S1]].sum():.0f}%")

## 11. Mapa interativo (para explorar no Colab)

Camadas do Earth Engine sobre imagem de satélite. Use o controle no canto superior direito para ligar e desligar. Os links dessas camadas expiram depois de algumas horas; o mapa de alertas da seção 12 é o que fica permanente.

In [ ]:
def camada_ee(mapa, imagem, vis, nome, mostrar=True):
    mid = ee.Image(imagem).getMapId(vis)
    folium.TileLayer(tiles=mid["tile_fetcher"].url_format, attr="Google Earth Engine",
                     name=nome, overlay=True, control=True, show=mostrar).add_to(mapa)


ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
PROB_FUSAO = PROBS["Fusão S1+S2"]
PRED_FUSAO = PROB_FUSAO.gte(LIMIAR_PROB).selfMask()

m = folium.Map(location=[LAT, LON], zoom_start=11, tiles=None)
folium.TileLayer(ESRI, attr="Esri", name="Satélite (Esri)").add_to(m)
vis_rgb = dict(min=0.0, max=0.18, gamma=1.2)
camada_ee(m, FEATURES, dict(bands=["r_t0", "g_t0", "b_t0"], **vis_rgb), f"Sentinel-2 antes ({ANO_ROTULO - 1})", False)
camada_ee(m, FEATURES, dict(bands=["r_t1", "g_t1", "b_t1"], **vis_rgb), f"Sentinel-2 depois ({ANO_ROTULO})")
camada_ee(m, FEATURES.select("vh_t1"), dict(min=-25, max=-10), f"Sentinel-1 VH depois ({ANO_ROTULO})", False)
camada_ee(m, LABEL.selfMask(), dict(palette=["ffd400"]), "Referência: perda florestal (Hansen)", False)
camada_ee(m, PROB_FUSAO, dict(min=0, max=1, palette=["1a9850", "fee08b", "d73027"]), "Probabilidade (fusão)", False)
camada_ee(m, PRED_FUSAO, dict(palette=["ff2d2d"]), "Detecção (fusão)")
folium.GeoJson(feicao(TESTE), name="Metade de teste",
               style_function=lambda f: {"color": "#ffffff", "weight": 2, "fill": False, "dashArray": "6 4"}).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

## 12. Alertas: da detecção ao polígono

Transforma os pixels detectados em polígonos, calcula a área em hectares, a probabilidade média e quanto de cada alerta coincide com a referência. Gera `alertas.geojson`, `alertas.csv` e `mapa_alertas.html` (permanente, abre em qualquer navegador).

In [ ]:
vetores = (PRED_FUSAO.toInt().reduceToVectors(
    geometry=AOI, crs=CRS, scale=ESCALA, geometryType="polygon", eightConnected=True,
    labelProperty="alerta", maxPixels=1e10, tileScale=4))


def com_area(f):
    return f.set("area_ha", f.geometry().area(1).divide(1e4))


vetores = vetores.map(com_area).filter(ee.Filter.gte("area_ha", AREA_MIN_HA))
atributos = PROB_FUSAO.addBands(LABEL.unmask(0).rename("ref"))
alertas_fc = (atributos.reduceRegions(collection=vetores, reducer=ee.Reducer.mean(),
                                      crs=CRS, scale=ESCALA, tileScale=4)
              .sort("area_ha", False).limit(2000))

feats = alertas_fc.getInfo()["features"]
assert feats, "Nenhum alerta gerado: reduza LIMIAR_PROB ou AREA_MIN_HA e rode de novo."
gdf = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")
gdf = gdf.rename(columns={"prob": "prob_media", "ref": "fracao_na_referencia"})
gdf.insert(0, "id_alerta", [f"A{i + 1:04d}" for i in range(len(gdf))])
centroides = gdf.to_crs(CRS).centroid.to_crs("EPSG:4326")
gdf["lon"], gdf["lat"] = centroides.x.round(5), centroides.y.round(5)
teste_shape = shape(TESTE.getInfo())
gdf["metade"] = np.where(centroides.within(teste_shape), "teste", "treino")
gdf["confirmado_pela_referencia"] = gdf["fracao_na_referencia"] >= 0.5
cols = ["id_alerta", "area_ha", "prob_media", "fracao_na_referencia", "confirmado_pela_referencia",
        "metade", "lon", "lat", "geometry"]
gdf = gdf[cols].round({"area_ha": 2, "prob_media": 3, "fracao_na_referencia": 3})

gdf.to_file("alertas.geojson", driver="GeoJSON")
gdf.drop(columns="geometry").to_csv("alertas.csv", index=False)

teste_alertas = gdf[gdf["metade"] == "teste"]
print(f"{len(gdf)} alertas (>= {AREA_MIN_HA} ha), somando {gdf['area_ha'].sum():,.0f} ha.")
print(f"Na metade de teste: {len(teste_alertas)} alertas; "
      f"{100 * teste_alertas['confirmado_pela_referencia'].mean():.0f}% confirmados pela referência.")
display(gdf.drop(columns="geometry").head(10))

In [ ]:
mapa_alertas = folium.Map(location=[LAT, LON], zoom_start=11, tiles=None)
folium.TileLayer(ESRI, attr="Esri", name="Satélite (Esri)").add_to(mapa_alertas)
folium.TileLayer("OpenStreetMap", name="Ruas (OSM)").add_to(mapa_alertas)
folium.GeoJson(feicao(AOI), name="Área-piloto",
               style_function=lambda f: {"color": "#ffffff", "weight": 2, "fill": False}).add_to(mapa_alertas)
folium.GeoJson(feicao(TESTE), name="Metade de teste",
               style_function=lambda f: {"color": "#ffffff", "weight": 1.5, "fill": False, "dashArray": "6 4"}).add_to(mapa_alertas)


def estilo_alerta(f):
    confirmado = f["properties"]["confirmado_pela_referencia"]
    return {"color": "#ff2d2d" if confirmado else "#ffb000", "weight": 1.5, "fillOpacity": 0.45}


folium.GeoJson(
    gdf.to_json(), name="Alertas de desmatamento", style_function=estilo_alerta,
    tooltip=folium.GeoJsonTooltip(
        fields=["id_alerta", "area_ha", "prob_media", "fracao_na_referencia", "metade"],
        aliases=["Alerta", "Área (ha)", "Probabilidade média", "Fração na referência", "Metade"]),
).add_to(mapa_alertas)

legenda = f'''
<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white; padding: 10px 12px;
            border-radius: 8px; font: 13px/1.4 sans-serif; box-shadow: 0 1px 4px rgba(0,0,0,.3)">
  <b>Alertas de desmatamento, {NOME_AOI}</b><br>
  Sentinel-1 + Sentinel-2, estação seca {ANO_ROTULO - 1} vs {ANO_ROTULO}<br>
  <span style="color:#ff2d2d">&#9632;</span> confirmado pela referência (Hansen)<br>
  <span style="color:#ffb000">&#9632;</span> não confirmado (a revisar)<br>
  Linha tracejada: metade usada só para teste
</div>'''
mapa_alertas.get_root().html.add_child(folium.Element(legenda))
folium.LayerControl(collapsed=False).add_to(mapa_alertas)
mapa_alertas.save("mapa_alertas.html")
mapa_alertas

## 13. Antes e depois do maior alerta da metade de teste

Gera a figura para a apresentação: óptico antes e depois, SAR antes e depois e a detecção sobreposta.

In [ ]:
alvo = (teste_alertas if len(teste_alertas) else gdf).sort_values("area_ha", ascending=False).iloc[0]
regiao = ee.Geometry(json.loads(json.dumps(mapping(alvo["geometry"])))).buffer(1500).bounds()


def miniatura(img_vis):
    url = img_vis.getThumbURL({"region": regiao, "dimensions": 512, "format": "png"})
    return Image.open(io.BytesIO(requests.get(url, timeout=120).content))


rgb_t0 = FEATURES.visualize(bands=["r_t0", "g_t0", "b_t0"], **vis_rgb)
rgb_t1 = FEATURES.visualize(bands=["r_t1", "g_t1", "b_t1"], **vis_rgb)
vh_t0 = FEATURES.select("vh_t0").visualize(min=-25, max=-10)
vh_t1 = FEATURES.select("vh_t1").visualize(min=-25, max=-10)
det = rgb_t1.blend(PRED_FUSAO.visualize(palette=["ff2d2d"], opacity=0.55))

paineis = [
    (rgb_t0, f"Óptico antes ({ANO_ROTULO - 1})"), (rgb_t1, f"Óptico depois ({ANO_ROTULO})"),
    (vh_t0, f"SAR VH antes ({ANO_ROTULO - 1})"), (vh_t1, f"SAR VH depois ({ANO_ROTULO})"),
    (det, "Detecção (fusão)"),
]
fig, axs = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, (img_vis, titulo) in zip(axs, paineis):
    ax.imshow(miniatura(img_vis))
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")
fig.suptitle(f"Alerta {alvo['id_alerta']}: {alvo['area_ha']:.1f} ha, probabilidade média {alvo['prob_media']:.2f}",
             fontsize=11, x=0.01, ha="left")
plt.tight_layout()
plt.savefig("antes_depois.png", bbox_inches="tight")
plt.show()

## 14. Resumo, README e pacote de resultados

Salva `resultados.json`, gera um `README.md` preenchido com os números desta execução e junta tudo em `resultados_piloto.zip`.

In [ ]:
fus, opt, sar = (RESULTADOS[k] for k in ["Fusão S1+S2", "Só óptico (S2)", "Só SAR (S1)"])
pior_mes = PIOR_MES
resumo = {
    "area_piloto": NOME_AOI, "centro_lon_lat": [LON, LAT], "lado_graus": LADO_GRAUS, "crs": CRS,
    "ano_rotulo": ANO_ROTULO, "janela_antes": T0, "janela_depois": T1,
    "imagens": n_imagens, "orbita_s1": PASSAGEM or "ambas",
    "pior_mes_optico": pior_mes, "cobertura_optica_pior_mes_pct": round(float(df_cob["optico"].min()), 1),
    "cobertura_sar_media_pct": round(float(df_cob["sar"].mean()), 1),
    "metricas_teste": {k: {m: (round(v, 4) if isinstance(v, float) else v) for m, v in r.items()}
                       for k, r in RESULTADOS.items()},
    "alertas_total": int(len(gdf)), "alertas_area_ha": round(float(gdf["area_ha"].sum()), 1),
    "alertas_teste_confirmados_pct": round(100 * float(teste_alertas["confirmado_pela_referencia"].mean()), 1)
    if len(teste_alertas) else None,
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

frase = (f"Na metade de teste, a fusão S1+S2 teve F1 de {fus['f1']:.3f} "
         f"(só óptico: {opt['f1']:.3f}; só SAR: {sar['f1']:.3f}). "
         f"Em {pior_mes}, o óptico observou {df_cob['optico'].min():.1f}% da área sem nuvem; "
         f"o SAR observou {df_cob['sar'].mean():.1f}% em média ao longo do ano.")
print(frase)

# Explora se falsos positivos do rótulo anual coincidem com perda Hansen registrada em 2024.
fp_mask = PROB_FUSAO.gte(LIMIAR_PROB).And(LABEL.eq(0))
lossyear = GFC.select("lossyear")
loss_2023 = LABEL.eq(1).reproject(crs=CRS, scale=ESCALA)
loss_2023_near = loss_2023.focalMax(kernel=ee.Kernel.square(radius=1, units="pixels"))
fp_2024 = fp_mask.And(lossyear.eq(24))
fp_near_2023 = fp_mask.And(loss_2023_near)
fp_neither = fp_mask.And(lossyear.neq(24)).And(loss_2023_near.Not())
fp_2024_only = fp_2024.And(loss_2023_near.Not())
fp_near_2023_only = fp_near_2023.And(lossyear.neq(24))
fp_overlap = fp_2024.And(loss_2023_near)
fp_counts = ee.Image.cat([
    fp_mask.rename("fp"),
    fp_2024.rename("fp_hansen_2024"),
    fp_near_2023.rename("fp_within_one_pixel_2023"),
    fp_neither.rename("fp_other"),
    fp_2024_only.rename("fp_2024_only"),
    fp_near_2023_only.rename("fp_near_2023_only"),
    fp_overlap.rename("fp_both_conditions"),
]).toInt().reduceRegion(
    reducer=ee.Reducer.sum().unweighted(), geometry=TESTE, crs=CRS,
    scale=ESCALA, maxPixels=1e10, tileScale=4,
).getInfo()
fp_total = int(fp_counts.get("fp", 0) or 0)
fp_2024_total = int(fp_counts.get("fp_hansen_2024", 0) or 0)
fp_near_total = int(fp_counts.get("fp_within_one_pixel_2023", 0) or 0)
fp_2024_only_total = int(fp_counts.get("fp_2024_only", 0) or 0)
fp_near_only_total = int(fp_counts.get("fp_near_2023_only", 0) or 0)
fp_other_total = int(fp_counts.get("fp_other", 0) or 0)
fp_overlap_total = int(fp_counts.get("fp_both_conditions", 0) or 0)

readme = f'''# Monitoramento de desmatamento com Sentinel-1 e Sentinel-2

Desenvolvi este estudo de caso pessoal ao longo de um fim de semana para demonstrar, de forma objetiva, meu interesse em contribuir com um time de visão computacional aplicada ao sensoriamento remoto. Investiguei quanto o radar Sentinel-1 pode acrescentar ao monitoramento de perda florestal quando imagens ópticas Sentinel-2 são limitadas por nuvens.

## Resultado principal

Na seca, o óptico já oferece observações úteis e a fusão acrescentou pouco à classificação: F1 de {fus['f1']:.3f} contra {opt['f1']:.3f} para o óptico. O valor operacional do radar aparece quando o óptico fica limitado: em {pior_mes}, o Sentinel-2 teve observação limpa em {df_cob['optico'].min():.1f}% da área, enquanto o Sentinel-1 observou {df_cob['sar'].mean():.1f}% em média ao longo do ano. O modelo só com SAR encontrou cerca de {sar['recall']:.0%} da perda Hansen na avaliação de seca (recall {sar['recall']:.3f}); o desempenho na estação chuvosa ainda não foi testado.

![Cobertura mensal de observação por sensor](cobertura_mensal.png)

## O que implementei

- Composição mediana de estação seca para Sentinel-2 SR Harmonized, com máscara Cloud Score+, e Sentinel-1 GRD (VV e VH), comparando {T0[0]} a {T0[1]} com {T1[0]} a {T1[1]}.
- Índices ópticos, variáveis SAR e diferenças entre os períodos.
- Rótulo Hansen Global Forest Change v1.12, considerando a floresta remanescente no início do período.
- Três Random Forests no Google Earth Engine: somente óptico, somente SAR e fusão.
- Holdout espacial: treino na metade {NOME_TREINO} e avaliação na metade {NOME_TESTE}, em vez de dividir aleatoriamente pixels vizinhos.
- Geração de alertas, imagens, tabelas, mapa HTML e pacote ZIP.

## Resultados na metade de teste

| Modelo | Precisão | Recall | F1 |
|---|---:|---:|---:|
| Só óptico (Sentinel-2) | {opt['precisao']:.3f} | {opt['recall']:.3f} | {opt['f1']:.3f} |
| Só SAR (Sentinel-1) | {sar['precisao']:.3f} | {sar['recall']:.3f} | {sar['f1']:.3f} |
| Fusão Sentinel-1 + Sentinel-2 | {fus['precisao']:.3f} | {fus['recall']:.3f} | {fus['f1']:.3f} |

![Comparação dos modelos](comparacao_modelos.png)
![Exemplo visual antes e depois](antes_depois.png)

O processamento gerou {len(gdf)} polígonos candidatos, somando aproximadamente {gdf['area_ha'].sum():,.0f} ha. Na metade de teste, {len(teste_alertas)} alertas foram comparados com o rótulo Hansen; {100 * teste_alertas['confirmado_pela_referencia'].mean():.0f}% tiveram pelo menos metade da área coincidente. Essa concordância não é confirmação independente de campo.

## Análise exploratória dos erros

Contei, sem ponderação de borda, os falsos positivos do modelo de fusão na metade de teste e comparei-os com a banda `lossyear` do Hansen e com a vizinhança dos pixels rotulados como perda em 2023. Dos {fp_total:,} pixels classificados como falso positivo frente ao rótulo de {ANO_ROTULO}, {fp_2024_total:,} têm `lossyear=24` (perda registrada pelo Hansen em 2024). Desses, {fp_overlap_total:,} também ficam a até um pixel de um pixel Hansen rotulado como perda de 2023; {fp_2024_only_total:,} têm perda registrada em 2024 fora dessa vizinhança. {fp_near_only_total:,} ficam perto de perda de 2023 sem `lossyear=24`, e {fp_other_total:,} não se enquadram em nenhuma das duas condições.

Esse padrão é compatível com um possível descompasso temporal do rótulo: parte dos falsos positivos pode corresponder a perda que o Hansen data em 2024, mas cuja mudança já era detectável nas imagens de 2023. A contagem não prova quando ocorreu a mudança nem confirma os pixels como desmatamento; a hipótese precisa de validação independente. Esta contagem inteira é exploratória e não é diretamente intercambiável com a matriz de confusão ponderada por área usada nas métricas principais.

## Reproduzir

### Pré-requisitos

- Python 3.10 ou superior e Git.
- Conta Google com acesso ao Earth Engine.
- Projeto Google Cloud autorizado para o uso pretendido no Earth Engine, com a Earth Engine API habilitada.
- Navegador na mesma máquina que executa o Jupyter, para concluir a autenticação local.

### Configurar o ambiente

Clone o repositório e crie um ambiente virtual:

```powershell
git clone https://github.com/laercio-siqueira-git/monitoramento-desmatamento-s1-s2.git
cd monitoramento-desmatamento-s1-s2
py -m venv .venv
.venv\\Scripts\\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt ipykernel
```

No macOS/Linux, use:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt ipykernel
```

Abra `piloto_desmatamento_s1_s2.ipynb` e selecione o kernel da pasta `.venv`. Também é possível usar Google Colab com um projeto Earth Engine válido.

### Configurar o Earth Engine e executar

Na seção **1. Autenticação no Earth Engine**, defina `PROJECT_ID` com um projeto Google Cloud que a pessoa executora possa usar. Para evitar gravar o ID no notebook, configure a variável de ambiente `EE_PROJECT` antes de iniciar o Jupyter:

```powershell
$env:EE_PROJECT = "ID_DO_PROJETO_DA_EQUIPE"
```

No macOS/Linux:

```bash
export EE_PROJECT="ID_DO_PROJETO_DA_EQUIPE"
```

Execute as células em ordem, de cima para baixo. Na primeira autenticação local, o navegador solicitará consentimento com escopo somente de leitura do Earth Engine. Confirme a conta e as permissões; não compartilhe tokens, códigos de autorização ou credenciais. A execução completa pode levar de 10 a 25 minutos.

## Parâmetros ajustáveis

Na seção **2. Parâmetros do piloto**, posso alterar `ANO_ROTULO`, `LADO_GRAUS`, `CS_LIMIAR`, `LIMIAR_PROB`, `AREA_MIN_HA` e `AOI_FIXA`. Alterações nesses parâmetros mudam os resultados; execute novamente o notebook e reporte os novos valores.

## Limitações e próximos passos

- Este experimento avalia uma área e um ano; não demonstra desempenho generalizável para toda a Amazônia.
- A comparação dos modelos usa composições da estação seca. A disponibilidade mensal de observações não prova que a fusão aumenta a acurácia na estação chuvosa.
- O rótulo Hansen tem resolução de 30 m e pode incluir perda por fogo. A validação com referência independente, como PRODES/DETER ou amostras revisadas, é um próximo passo.
- Os pontos de treino são balanceados por classe; as probabilidades não devem ser interpretadas automaticamente como probabilidades calibradas para a prevalência real de desmatamento.
- Os polígonos são alertas candidatos para revisão, não confirmações de campo.

## Dados e créditos

- Sentinel-1 e Sentinel-2: dados Copernicus processados no Google Earth Engine.
- Cloud Score+: Pasquarella et al. (2023), CVPR Workshops.
- Hansen Global Forest Change v1.12: Hansen et al. (2013), *High-Resolution Global Maps of 21st-Century Forest Cover Change*, *Science* 342, 850–853.

## Autor

Laércio Siqueira, cientista de dados.
'''
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

arquivos = ["resultados.json", "README.md", "alertas.geojson", "alertas.csv", "mapa_alertas.html",
            "cobertura_mensal.png", "comparacao_modelos.png", "importancia_variaveis.png", "antes_depois.png"]
with zipfile.ZipFile("resultados_piloto.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for a in arquivos:
        try:
            z.write(a)
        except FileNotFoundError:
            print("Não encontrado (pulei):", a)

try:
    from google.colab import files
    files.download("resultados_piloto.zip")
except ImportError:
    print("Fora do Colab: o pacote está em resultados_piloto.zip")